# Train path classification model

In [13]:
import torch

print("PyTorch version:", torch.__version__)
print("is cuda available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch version: 2.7.0+cu128
is cuda available: True


In [14]:
import os

data_dir = os.path.abspath('../data/FIVES')
train_split = 'train_clean'

## Overview of the training pipeline and it's main modules

![train_pipeline](../images/train_pipeline.png "Train pipeline diagram")

## Features Generator / Features extractor
The features generator is here a pretrained UNet (see [U-Net pretraining notebook (2)](./02_pretrain_unet.ipynb))

We load the checkpoint of the pretrained UNet and use it as a features generator for our path classification model.

We replace the last 32 to 1 layers convolutional layer of the pretrained UNet by a new one with 32 output channels, and we keep the pretrained weights for the rest of the UNet. We also set `freeze_pretrained` to False to allow fine-tuning of the pretrained UNet during the training of the path classification model.

In [15]:
from path_neural_networks.models.features_generators import FeaturesGenerator, PretrainedUnetFeaturesGenerator
from utils.other import pretty_dict_print

unet_checkpoint_dir = os.path.abspath('../checkpoints/unet_pretraining')
unet_ckpt_path = os.path.join(unet_checkpoint_dir, os.listdir(unet_checkpoint_dir)[0])
print("Using UNet checkpoint:", unet_ckpt_path)

features_generator_out_channels = 32

features_generator: FeaturesGenerator = PretrainedUnetFeaturesGenerator(
    ckpt_path=unet_ckpt_path,
    device=device,
    out_channels=features_generator_out_channels,
    freeze_pretrained=False,
    skip_connection=False
)
print("Features generator configuration:")
pretty_dict_print(features_generator.as_dict())

Using UNet checkpoint: /home/morand/afs/EVAPORE/checkpoints/unet_pretraining/best-checkpoint-epoch=65-val_loss=0.0685.ckpt
Features generator configuration:
{
    cls: PretrainedUnetFeaturesGenerator
    out_channels: 32
    skip_connection: False
    ckpt_path: /home/morand/afs/EVAPORE/checkpoints/unet_pretraining/best-checkpoint-epoch=65-val_loss=0.0685.ckpt
    device: cuda
    freeze_pretrained: False
}


/home/morand/afs/EVAPORE/.venv/lib/python3.12/site-packages/torchmetrics/utilities/prints.py:43: UserWarning: DiceScore metric currently defaults to `average=micro`, but will change to`average=macro` in the v1.9 release. If you've explicitly set this parameter, you can ignore this warning.
  warnings.warn(*args, **kwargs)


## Path Sampler
For each path if the image, the path sampler will sampler a set squares patches of different sizes, for each coordinate in the path, and aggregate the features in these patches using the specified method (e.g. max pooling).

Using multiple square sizes allows to capture features at different scales, which can be beneficial for vessel segmentation where vessels can have varying widths, and because our vessels are note perfectly centered on the euclidean minimum path, so we want to capture features in a larger area around the path coordinates.

The aggregation method allows to summarize the features in the sampled patches into a single feature vector for each path coordinate, which can then be used as input to the path neural network.

At the end, we get a tensor of shape (n_features * n_scales, path_length) for each path, where n_features is the number of output channels of the features generator, n_scales is the number of different square sizes used for sampling, and path_length is the number of coordinates in the path.

![path_sampler](../images/path_features_sampling.png)

In [16]:
from path_neural_networks.models.path_samplers import PathSampler, MultiScaleSquarePathSampling, SamplingMaxAggregation

sampling_square_sizes = [1, 3, 5]
sampling_aggregation_method = SamplingMaxAggregation()

path_sampler: PathSampler = MultiScaleSquarePathSampling(
    in_channels=features_generator_out_channels,
    square_sizes=sampling_square_sizes,
    aggregation=sampling_aggregation_method
)
print("Path sampler configuration:")
pretty_dict_print(path_sampler.as_dict())

Path sampler configuration:
{
    cls: MultiScaleSquarePathSampling
    in_channels: 32
    out_channels: 96
    square_sizes: [1, 3, 5]
    n_scale: 3
    aggregation: {
        cls: SamplingMaxAggregation
    }
}


## Path encoder

The path encoder is a convolutional neural network that takes as input the features sampled along the path by the path sampler, and encodes them into a fixed-size feature vector that can be used for classification.

It is composed of a series of convolutional layers, followed by a global pooling operation (here a max pooling) to aggregate the features along the path, and optionally skip connections and residual blocks to improve the flow of information and gradients through the network.

At the end, we get a feature vector of size 256 for each path, which can then be used as input to a classifier to predict the class of the path (e.g. true vessel or false positive).

![path_encoder](../images/path_encoder.png "Path encoder diagram")

In [17]:
from path_neural_networks.models.path_encoders import PathEncoder, ConvMaxPoolingPathEncoder

conv_path_residual_blocks = False
conv_path_skip_connections = False
conv_path_layers = [None, None, 256]

path_encoder: PathEncoder = ConvMaxPoolingPathEncoder(
    in_channels=path_sampler.out_channels, 
    hidden_layers=conv_path_layers, 
    skip_connection=conv_path_skip_connections, 
    residual_blocks=conv_path_residual_blocks
)
print("Path encoder configuration:")
pretty_dict_print(path_encoder.as_dict())

Path encoder configuration:
{
    cls: ConvMaxPoolingPathEncoder
    in_channels: 96
    hidden_layers: [96, 192, 256]
    kernel_size: 3
    padding: 1
    residual_blocks: False
    skip_connection: False
    pooling_operation: {
        cls: MaxPooling
        out_channels_factor: 1
    }
    out_channels: 256
}


## Path classifier

The path classifier is a fully connected network that takes as input the feature vector produced by the path encoder for each path, and outputs a binary classification (e.g. true vessel or false positive).

It is composed of a series of fully connected layers, optionally with dropout for regularization, ReLU activations, and layer normalization to improve training stability and performance.

The number of hidden layers and their sizes can be tuned to find the best architecture for the task at hand.

In [18]:
from path_neural_networks.models.path_classifiers import PathClassifier, FCNPathClassifier

path_classifier_n_hidden_layers = 2
path_classifier_dropout = 0

path_classifier: PathClassifier = FCNPathClassifier(
    in_channels=path_encoder.out_channels,
    n_hidden_layers=path_classifier_n_hidden_layers,
    num_classes=1,
    dropout=path_classifier_dropout
)
print("Path classifier configuration:")
pretty_dict_print(path_classifier.as_dict())

Path classifier configuration:
{
    cls: FCNPathClassifier
    in_channels: 256
    n_hidden_layers: 2
    dropout: [0.0, 0.0]
    num_classes: 1
}


## Loading data (images, centerlines, ground truths...)

In [19]:
max_dist = 100

if max_dist is None:
    centerlines_dirname = "euclidean_all_centerlines"
else:
    centerlines_dirname = f"euclidean_lt_{max_dist}_centerlines"

In [20]:
from path_neural_networks.data.image_centerline_dataset import ImageCenterlineDataset
from path_neural_networks.data.image_centerline_datamodule import ImageCenterlineDatamodule
import albumentations as A
from albumentations.pytorch import ToTensorV2
from utils.data_augmentation.add_gaussian_noise import AddGaussNoise

split_file_path=os.path.join(data_dir, "splits.json")
val_split_ratio = 0.2
use_foreground_pixels_only_for_normalization = True
data_seed = 42
split_seed = 42
shuffle_train = True

dataset = ImageCenterlineDataset(data_dir=data_dir, centerline_dirname=centerlines_dirname)
datamodule = ImageCenterlineDatamodule(dataset=dataset, 
                                        split_file_path=split_file_path,
                                        train_split_name=train_split,
                                        val_split_ratio=val_split_ratio,
                                        train_transforms=None,
                                        val_transforms=None,
                                        test_transforms=None,
                                        seed = split_seed,
                                        shuffle_train = shuffle_train)
datamodule.setup()

stats = dataset.get_dataset_stats(split_name='train_clean', split_indices=datamodule.train_indices.tolist() + datamodule.val_indices.tolist())
if use_foreground_pixels_only_for_normalization:
    stats = stats['foreground']
else:
    stats = stats['full_image']
mean, std = stats['mean'], stats['std']
print("Dataset stats used for normalization:")
pretty_dict_print(stats)

Dataset split: Train=440, Val=109, Test=200
Loading dataset stats for split 'train_clean' from /home/morand/afs/EVAPORE/data/FIVES/image_stats.json...
Dataset stats used for normalization:
{
    mean: [0.44817834316662786, 0.20165531537789622, 0.08165605062841931]
    std: [0.17302453064340234, 0.1008530268569081, 0.05196561252496086]
}


### Adding data augmentation

In [21]:
use_data_augmentation = True

if use_data_augmentation:
        train_transforms = A.Compose([
            A.RandomBrightnessContrast(
            brightness_limit=(-0.15, 0.15),
            contrast_limit=(-0.15, 0.15),
            p=0.5
        ),
        A.Lambda(image=AddGaussNoise(std=(0.005, 0.015)), p=0.5),
        A.Normalize(mean=mean, std=std, max_pixel_value=1.0),
        ToTensorV2()
    ], seed=data_seed)
else:
    train_transforms = A.Compose([
        A.Normalize(mean=mean, std=std, max_pixel_value=1.0),
        ToTensorV2()
    ], seed=data_seed)

val_transforms = A.Compose([
    A.Normalize(mean=mean, std=std, max_pixel_value=1.0),
    ToTensorV2()
], seed=data_seed)

datamodule = ImageCenterlineDatamodule(dataset=dataset,
                                        split_file_path=split_file_path,
                                        train_split_name=train_split,
                                        val_split_ratio=val_split_ratio,
                                        train_transforms=train_transforms,
                                        val_transforms=val_transforms,
                                        test_transforms=val_transforms,
                                        seed = split_seed,
                                        shuffle_train = shuffle_train)

### Initialize the loss function

In [22]:
from path_neural_networks.models.losses import PathClassificationLoss, WeightedBCEWithLogitsLoss, BCEWithLogitsLoss

use_pos_weight_in_loss = True

loss_fn: PathClassificationLoss
if use_pos_weight_in_loss:
    classes_stats = dataset.get_dataset_classes_stats()
    classes_ratio = classes_stats['classes_ratio']
    loss_fn = WeightedBCEWithLogitsLoss(classes_ratio=classes_ratio)
else:
    loss_fn = BCEWithLogitsLoss()

## Initialize the model with all the previous components

In [23]:
from path_neural_networks.models import ReducedPipelineLitModule
from path_neural_networks.utils.symmetry_enforcement import SymmetryEnforcementMode

learning_rate = 3e-4
metrics = ["accuracy", "auroc", "recall", "precision", "pr_auc"]
symmetry_enforcement_mode = SymmetryEnforcementMode.NONE

model = ReducedPipelineLitModule(
    features_generator=features_generator,
    path_sampler=path_sampler,
    path_encoder=path_encoder,
    path_classifier=path_classifier,
    edge_classification_loss_fn=loss_fn,
    metrics=metrics,
    lr=learning_rate,
    symmetry_enforcement_mode=symmetry_enforcement_mode
)

In [ ]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

checkpoint_dir = os.path.abspath('../checkpoints/main_model_training')
os.amakedirs(checkpoint_dir, exist_ok=True)

callbacks = [
    ModelCheckpoint(
        dirpath=checkpoint_dir,
        monitor="val_pr_auc",
        mode="max", 
        save_top_k=1, 
        filename="best-pr-auc-checkpoint-{epoch:02d}-{val_pr_auc:.4f}"
    ),
    EarlyStopping(
        monitor="val_pr_auc",
        patience=20,
        min_delta=1e-3,
        verbose=True,
        mode="max"
    ),
]

In [25]:
from pytorch_lightning import Trainer
from pytorch_lightning.loggers import CSVLogger

logger = CSVLogger(".")

torch.set_float32_matmul_precision("medium")
trainer = Trainer(accelerator='gpu', 
                  devices="auto",
                  num_nodes=1,
                  max_epochs=100,
                  precision="16-mixed",
                  detect_anomaly=False, 
                  callbacks=callbacks,
                  logger=logger,
                  gradient_clip_val=1.0,
                  gradient_clip_algorithm="norm",
                  val_check_interval=0.1
)
print(trainer)
print("Num GPUs:", trainer.num_devices)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Num GPUs: 1


In [26]:
trainer.fit(model, datamodule=datamodule)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                        | Type                            | Params | Mode 
----------------------------------------------------------------------------------------
0 | features_generator          | PretrainedUnetFeaturesGenerator | 4.9 M  | train
1 | path_sampler                | MultiScaleSquarePathSampling    | 0      | train
2 | path_encoder                | ConvMaxPoolingPathEncoder       | 512 K  | train
3 | path_classifier             | FCNPathClassifier               | 41.6 K | train
4 | edge_classification_loss_fn | WeightedBCEWithLogitsLoss       | 0      | train
5 | train_metrics               | ModuleDict                      | 0      | train
6 | val_metrics                 | ModuleDict                      | 0      | train
7 | test_metrics                | ModuleDict                      | 0      | train
----------------------------------------------------------------------------------------
5.5 M     Trainable params
0    

Dataset split: Train=440, Val=109, Test=200
Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/morand/afs/EVAPORE/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


/home/morand/afs/EVAPORE/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Epoch 0:  10%|█         | 44/440 [06:46<1:00:57,  0.11it/s, v_num=271]

...

Epoch 3: 100%|██████████| 440/440 [1:15:33<00:00,  0.10it/s, v_num=271, val_loss=0.164, val_accuracy=0.945, val_auroc=0.987, val_recall=0.945, val_precision=0.926, val_pr_auc=0.983, lr=0.0003, train_loss=0.183, train_accuracy=0.941, train_auroc=0.983, train_recall=0.945, train_precision=0.917, train_pr_auc=0.973]

Metric val_pr_auc improved by 0.001 >= min_delta = 0.001. New best score: 0.984


Epoch 5: 100%|██████████| 440/440 [1:15:33<00:00,  0.10it/s, v_num=271, val_loss=0.174, val_accuracy=0.944, val_auroc=0.987, val_recall=0.934, val_precision=0.933, val_pr_auc=0.983, lr=0.000299, train_loss=0.158, train_accuracy=0.949, train_auroc=0.987, train_recall=0.950, train_precision=0.930, train_pr_auc=0.977]

Monitored metric val_pr_auc did not improve in the last 20 records. Best score: 0.984. Signaling Trainer to stop.


Epoch 5: 100%|██████████| 440/440 [1:16:22<00:00,  0.10it/s, v_num=271, val_loss=0.171, val_accuracy=0.944, val_auroc=0.987, val_recall=0.934, val_precision=0.933, val_pr_auc=0.983, lr=0.000298, train_loss=0.144, train_accuracy=0.954, train_auroc=0.990, train_recall=0.952, train_precision=0.938, train_pr_auc=0.986]
